# A tour of Retentioneering 5

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/retentioneering/retentioneering-tools/blob/master/notebooks/retentioneering_5_tour.ipynb)

[Retentioneering](https://github.com/retentioneering/retentioneering-tools) is an open-source Python library for user path analysis: a code-first take on the kind of analysis you would do in Amplitude or Mixpanel, running on your own data.

This notebook runs as is in Google Colab, no data to upload. It uses the bundled e-commerce dataset: six months of a consumer-electronics store, 600 users, ~3,600 sessions, ~31,000 events. Every chart is interactive: click nodes and edges, drag, open the sidebar (the panel icon in the top-right corner) to change parameters live.

Docs: [quick start](https://retentioneering.com/docs/quick-start) · [widgets](https://retentioneering.com/docs/widgets) · [path patterns](https://retentioneering.com/docs/path-patterns)

In [ ]:
%pip install -q retentioneering

In [ ]:
import retentioneering as rete

df = rete.datasets.load_ecom(as_dataframe=True)

stream = rete.Eventstream(
    df,
    {
        "path_cols": ["user_id", "session_id"],
        "segment_cols": [
            "platform",
            "acquisition_channel",
            "user_cohort",
            "user_lifecycle",
        ],
    },
)
df.head()

## 0. The big picture

A transition graph of all sessions, with the path to purchase highlighted. Click **Default** above the graph to see all transitions, click any node to focus it.

In [ ]:
path = [
    "catalog",
    "product_view",
    "add_to_cart",
    "cart",
    "shipping_details",
    "payment_details",
    "purchase",
]

# pin the path to purchase to hand-picked positions; every other event keeps its automatic place
positions = {
    "catalog": {"x": -64, "y": 227},
    "product_view": {"x": -263, "y": 15},
    "add_to_cart": {"x": -277, "y": -165},
    "cart": {"x": 35, "y": -245},
    "shipping_details": {"x": 223, "y": -204},
    "payment_details": {"x": 257, "y": 72},
    "purchase": {"x": 111, "y": 242},
}

stream.transition_graph(
    path_col="session_id",
    views=[
        {
            "name": "Path to purchase",
            "focus": {"type": "path", "nodes": path},
            "hiddenEvents": ["path_start", "path_end"],
            "nodePositions": positions,
            "viewport": "fit",
        }
    ],
    view="Path to purchase",
    sidebar_open=False,
)

## 1. Compare groups of paths

The schema declares two path levels: a user and a session. Most examples in this notebook analyse **sessions** (`path_col="session_id"`), because a checkout is something that happens within one visit.

In diff mode every widget shows where two groups of paths behave differently: red means the first group does it more, blue means less. The groups don't have to be columns you already have: a segment can be defined on the fly.

### Root cause of a metric drop: before vs. after

Purchase conversion dropped between May 19 and Jun 7. Mark that window as a segment and compare it with the rest of the time. The graph opens focused on the payment step; click **Default** above the graph to see everything.

In [ ]:
incident = stream.add_segment("incident", time_range=("2024-05-19", "2024-06-07"))

incident.transition_graph(
    diff=["incident", "inside", "outside"],
    path_col="session_id",
    views=[
        {
            "name": "Payment step",
            "focus": {"type": "node", "event": "payment_details"},
            "edgeFilter": {"mode": "topk", "k": 3},
        }
    ],
    view="Payment step",
    sidebar_open=False,
)

During the incident the transition from `payment_details` to `purchase` fell by 13 p.p., and a new route appeared: `payment_details → payment_error → support_chat`. Demand didn't drop, payments did.

### Funnels: users who did X and Y but never reached Z

`funnel_events` labels each path with the deepest funnel step it completed in order. Compare the sessions that got as far as `shipping_details` but never purchased with the ones that did.

In [ ]:
funnel = stream.add_segment(
    "funnel",
    funnel_events=["cart", "shipping_details", "purchase"],
    path_col="session_id",
)

funnel.transition_graph(
    diff=["funnel", "shipping_details", "purchase"],
    path_col="session_id",
    views=[
        {
            "name": "Payment step",
            "focus": {"type": "node", "event": "payment_details"},
            "edgeFilter": {"mode": "topk", "k": 3},
        }
    ],
    view="Payment step",
    sidebar_open=False,
)

The sessions that stalled rarely come back to `payment_details` after an error or a support chat: the ones that bought mostly retried.

### Newcomers vs. regulars

The same comparison works for any segment column. Here new users vs. loyal ones, step by step from the start of a session.

In [ ]:
stream.step_matrix(
    diff=["user_lifecycle", "new", "loyal"],
    path_col="session_id",
    max_steps=8,
    sidebar_open=False,
)

## 2. Slice paths with patterns

Path patterns are a small regex-like language over events: `.*` is any run of events, `[a|b]` is one of the listed events, `[^a|b]*` is a run with none of them, `path_start` / `path_end` are the path's boundaries. See [Path Patterns](https://retentioneering.com/docs/path-patterns) for the full syntax.

This pattern finds sessions where the user opened the cart and then left without starting checkout or asking support, a silent cart abandonment:

In [ ]:
pattern = "cart->[^shipping_details|support_chat]*->path_end"

abandoned = stream.filter_paths(
    {
        "op": "=",
        "metric": "matches_pattern",
        "value": True,
        "metric_args": {"pattern": pattern},
    },
    path_col="session_id",
)
print(
    f"{abandoned.df['session_id'].nunique()} of {stream.df['session_id'].nunique()} sessions match"
)

Patterns also mark where to cut paths. Keep only the part of each session that starts at a failed payment and ends at the support chat that followed without a retry:

In [ ]:
after_error = stream.truncate_paths(
    start_anchor={"pattern": "payment_details->payment_error"},
    end_anchor={"pattern": "payment_error->[^payment_details]*->support_chat"},
    path_col="session_id",
)
after_error.to_dataframe().head(10)

### Zoom in on every funnel step

In Step Matrix and Step Sankey a pattern sets the anchors. Here each of the two funnel events gets its own two-step neighborhood, and whatever happened between them is folded into the gap:

In [ ]:
stream.step_sankey(
    path_pattern="shipping_details->.*->payment_details",
    step_window=2,
    path_col="session_id",
    sidebar_open=False,
)

## 3. Behavioral clusters

Cluster Analysis groups paths by behavior and shows what makes each cluster different. Here sessions are described by their length and by how often each event occurs: each row of the heatmap is an event's average count per session, and the color shows where a cluster stands out. The best number of clusters is picked automatically by silhouette score. The **Silhouette** tab shows how well the data splits for each number of clusters; click a bar to inspect that partition.

In [ ]:
stream.cluster_analysis(
    features=[{"metric": "length"}, {"metric": "event_count_bulk"}],
    method_args={"n_clusters": "3-6"},
    path_col="session_id",
    sidebar_open=False,
)

To keep working with the clusters, turn them into a segment column with `add_clusters` and use them in diff mode like any other segment.

## 4. Hand the analysis to an AI agent

Retentioneering ships a local MCP server (in beta). Start it from a notebook on your machine and connect Claude Code, Codex, Cursor or another MCP client:

```python
rete.mcp.serve(stream, context={"description": "E-commerce store. Main KPI: purchase conversion."})
```

The agent calls the library's tools and builds an interactive HTML report in which every number links to the chart it came from. The MCP server needs a local Python environment, so it doesn't run in Colab. See the [MCP server guide](https://retentioneering.com/docs/mcp-server).

## Next steps

- Try it on your own data: any table with a user id, an event name and a timestamp will do. Start with the [quick start](https://retentioneering.com/docs/quick-start).
- Every widget exports to a standalone interactive HTML file: `widget.export_html("report.html")`.
- Questions, bugs, ideas: [GitHub issues](https://github.com/retentioneering/retentioneering-tools/issues).